# Building Dimensions table for the races and circuits silver tables

### Getting the batch id as input parameter

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00.Common/01.Environment-config

In [0]:
%run "../00.Common/04.Helper_Notebook_Gold"

### Assinging variable names

In [0]:
target_name = f"{catalog_name}.{gold_schema}.dim_races"
races_table = f"{catalog_name}.{silver_schema}.races"
circuit_table = f"{catalog_name}.{silver_schema}.circuits"

### Reading races and circuits tables from silver schema

In [0]:
#import the sql function and filter the df with the batch id
from pyspark.sql import functions as F
races_df = spark.table(races_table).filter(F.col("batch_id")==v_batch_id)
circuits_df = spark.table(circuit_table).filter(F.col("batch_id")==v_batch_id)

### Join the races and circuits df and select the required columns

In [0]:
dim_races_df = (
    races_df.join(
        circuits_df, races_df.circuit_id == circuits_df.circuits_id, "inner"
        ).select(races_df.season,
                 races_df.round,
                 races_df.race_name,
                 races_df.race_date,
                 circuits_df.circuit_name,
                 circuits_df.locality,
                 circuits_df.country)
)

In [0]:
display(dim_races_df)

In [0]:
dim_races_df.columns

### Write the final dataframe into the gold schema as delta table

In [0]:
write_to_gold(
    input_df = dim_races_df,
    target_table = target_name,
    merge_condition = "t.season=s.season AND t.round=s.round",
    columns_to_update = ['season',
 'round',
 'race_name',
 'race_date',
 'circuit_name',
 'locality',
 'country']
)

In [0]:
display(spark.table(target_name))